## Section-B – Data Cleaning
### Introduction

Raw business data often contains missing values, inconsistent formats, invalid records, and referential integrity issues. Before loading data into a database or performing analytics, these issues must be identified and corrected.

This notebook performs the data cleaning and validation phase for the E-Commerce Order Analytics System project.

### Objectives
- Load raw CSV files.
- Clean and standardize order dates.
- Handle missing customer IDs.
- Normalize product names.
- Validate customer email addresses.
- Check referential integrity between orders and order_items.
- Save cleaned datasets.
- Generate a summary report of data quality issues.

## Import Libraries

In [4]:
import pandas as pd
import numpy as np 
import os 
from datetime import datetime

## Load Raw Datasets

In [5]:
orders_df = pd.read_csv('data/raw-data/orders.csv')
order_items_df = pd.read_csv('data/raw-data/order_items.csv')
products_df = pd.read_csv('data/raw-data/products.csv')
customers_df = pd.read_csv('data/raw-data/customers.csv')

print('Datasets loaded successfully')

Datasets loaded successfully


## Preview Raw Data

In [6]:
orders_df.head()
products_df.head()
customers_df.head()

,customer_id,customer_name,email,registration_date,customer_type
0,1,Marisa Wright,jlogan@example.net,2025-07-10,PREMIUM
1,2,Sonya Mccormick,erodriguez@example.net,2025-02-03,VIP
2,3,Kellie Knox,lgrimes@example.net,2024-10-28,REGULAR
3,4,Shane Aguilar,riverajonathan@example.com,2024-10-23,VIP
4,5,Edwin Delgado,nathan92@example.com,2025-08-15,PREMIUM


## 1. Clean Orders
### Objective
- Fix incorrect date formats.
- Handle missing customer IDs.
## Function

In [7]:
def clean_orders(df):
    # Count missing customer IDs before cleaning
    missing_customer_ids = df['customer_id'].isna().sum()
    
    # Replace missing customer IDs with -1
    df['customer_id'] = df['customer_id'].fillna(-1)
    
    # Function to parse multiple date formats
    def parse_date(x):
        for fmt in ('%Y-%m-%d %H:%M:%S', '%d-%m-%Y %H:%M:%S'):
            try:
                return datetime.strptime(str(x), fmt)
            except:
                continue 
        return pd.NaT 
    
    df['order_date'] = df['order_date'].apply(parse_date)
    invalid_dates = df['order_date'].isna().sum()
    return df, missing_customer_ids, invalid_dates

## Apply Cleaning

In [8]:
orders_clean, missing_customer_ids, invalid_dates = clean_orders(orders_df)
print('Missing customer IDs handled:', missing_customer_ids) 
print('Invalid dates found:', invalid_dates)

Missing customer IDs handled: 12
Invalid dates found: 0


# 2. Clean Products
## Objective

### Normalize product names by:

- Removing extra spaces.
- Converting to title case.
### Function

In [9]:
def clean_products(df):
    df['product_name'] = (
        df['product_name'] 
        .astype(str)
        .str.strip()
        .str.title() 
    ) 
    return df

### Apply Cleaning

In [10]:
products_clean = clean_products(products_df) 
products_clean.head()

,product_id,product_name,category,subcategory,cost_price
0,P0001,Forward Phone,Electronics,Phone,1262.21
1,P0002,Sort Shirt,Clothing,Shirt,4466.29
2,P0003,Probably Headphones,Electronics,Headphones,513.79
3,P0004,Some Shirt,Clothing,Shirt,3594.30
4,P0005,College Biography,Books,Biography,1427.04


# 3. Validate Emails
## Objective

Identify customers with invalid email addresses.

### Function

In [13]:
def validate_emails(df):
    invalid_mask = ~df['email'].astype(str).str.contains(
        r'^[^@]+@[^@]+\\.[^@]+$',
        regex=True, na=False
    )
    return df.loc[invalid_mask, 'customer_id'].tolist()

### Apply Validation

In [14]:
invalid_email_ids = validate_emails(customers_df)
print('Invalid email count:', len(invalid_email_ids)) 
print(invalid_email_ids[:10])

Invalid email count: 500
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


# 4. Check Referential Integrity
## Objective

Find order items whose order_id does not exist in the orders table.

### Function

In [15]:
def check_referential_integrity(orders, order_items): 
    invalid_items = order_items[ 
        ~order_items['order_id'].isin(orders['order_id']) 
    ]
    return invalid_items

### Apply Validation

In [16]:
invalid_order_items = check_referential_integrity(
    orders_clean,
    order_items_df
)
print('Invalid order item references:', len(invalid_order_items))
invalid_order_items.head()

Invalid order item references: 0


,item_id,order_id,product_id,quantity,unit_price,discount_percent


# 5. Identify Negative Quantities
## Objective

Find order items with negative quantities.

### Function

In [17]:
negative_quantity_count = (
    order_items_df['quantity'] < 0
).sum()
print('Negative quantity rows:', negative_quantity_count)

Negative quantity rows: 48


# 6. Save Cleaned Datasets

In [18]:
os.makedirs('data/cleaned-data', exist_ok=True)

orders_clean.to_csv(
    'data/cleaned-data/orders_clean.csv',
    index=False
)

order_items_df.to_csv(
    'data/cleaned-data/order_items_clean.csv',
    index=False
)

products_clean.to_csv(
    'data/cleaned-data/products_clean.csv',
    index=False
)

customers_df.to_csv(
    'data/cleaned-data/customers_clean.csv',
    index=False
)

print('Cleaned CSV files saved successfully')

Cleaned CSV files saved successfully


# 7. Generate Issues Report

In [20]:
issues_report = {
    'missing_customer_ids': int(missing_customer_ids), 
    'invalid_dates': int(invalid_dates),
    'invalid_emails': int(len(invalid_email_ids)),
    'invalid_order_item_references': int(len(invalid_order_items)),
    'negative_quantity_rows': int(negative_quantity_count) 
}

issues_df = pd.DataFrame(
    issues_report.items(),
    columns=['Issue', 'Count']
)

issues_df

,Issue,Count
0,missing_customer_ids,12
1,invalid_dates,0
2,invalid_emails,500
3,invalid_order_item_references,0
4,negative_quantity_rows,48


# 8. Save Issues Report

In [22]:
issues_df.to_csv(
    'reports/issues_report.csv',
    index=False
) 
print('Issues report saved successfully')

Issues report saved successfully


### Conclusion

In this notebook, the raw e-commerce datasets were successfully cleaned and validated. Missing customer IDs were handled, inconsistent date formats were standardized, product names were normalized, invalid email addresses were identified, and referential integrity between orders and order items was verified.

The cleaned datasets were saved in the data/cleaned directory, and a data quality report was generated summarizing all detected issues. These cleaned datasets are now ready for database loading and SQL-based business analytics in the next phase of the project.